In [ ]:
import sys
sys.path.append("../")
import utils

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter('ignore')

# Parameters (change as needed)
csv_path = "../data/dataset_1.csv"   # adjust path as needed
seed = 0
n_init = 10
n_steps = 50
shuffle_seed = 0

# Load data
df_ = pd.read_csv(csv_path)

# Use last column as target (prop)
prop = list(df_.columns)[-1]

# Standardize explanatory features (all columns except the last)
df_.iloc[:, :-1] = StandardScaler().fit_transform(df_.iloc[:, :-1])

# Split target and features
df_oracle = df_[[prop]]
df_desc = df_.drop(prop, axis=1)

# Initial labeled sample
idx_sample = list(df_oracle.sample(n_init, random_state=seed).index)
df = df_oracle.iloc[idx_sample].reset_index(drop=True)
df_X = df_desc.iloc[idx_sample].reset_index(drop=True)

# Pool (remaining)
df_pool = df_oracle.drop(idx_sample).reset_index(drop=True)
df_pool_X = df_desc.drop(idx_sample).reset_index(drop=True)

# Shuffle pool (to mimic original shuffling behavior)
idx = list(df_pool_X.sample(frac=1, random_state=shuffle_seed).index)
df_pool = df_pool.iloc[idx].reset_index(drop=True)
df_pool_X = df_pool_X.iloc[idx].reset_index(drop=True)

# Records
obs_max = [df[prop].max()]

# Main loop: randomly sample one point from pool per iteration
for num in range(n_steps):
    # Deterministic random selection using seed offset by iteration count
    sampled_label = df_pool.sample(1, random_state=seed + num).index[0]
    # get positional index corresponding to the sampled label (robust even if index is not RangeIndex)
    sampled_pos = df_pool.index.get_loc(sampled_label)

    # Move selected sample from pool to labeled set (features and targets)
    df_X = pd.concat([df_X, df_pool_X.iloc[[sampled_pos]]]).reset_index(drop=True)
    df_pool_X = df_pool_X.drop(sampled_label).reset_index(drop=True)

    df = pd.concat([df, df_pool.iloc[[sampled_pos]]]).reset_index(drop=True)
    df_pool = df_pool.drop(sampled_label).reset_index(drop=True)

    # Record observed max
    obs_max.append(df[prop].max())